In [2]:
import sys
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

from pyproj import CRS, Transformer
from scipy.spatial.distance import pdist, squareform

import pyvista as pv

from scipy.spatial.distance import pdist, squareform
from scipy.interpolate import RBFInterpolator

# Inputs

In [3]:
#Defining path to file
path = "/mnt/c/Users/josem/OneDrive/Documents/Dummy_data/new_dummy_data3.csv" # r'/path' for windows
df = pd.read_csv(path)

# File path for the model domain which will be used later
boundary_path = ("/mnt/c/Users/josem/OneDrive/Documents/Dummy_data/model_domain/shape_file/model_domain_2/model_domain_2.shp")



# Validation and test functions

In [ ]:
# CRS
def validate_projection(df_, X1, Y1, source_crs, model_crs,):
    
    """Verify projected X/Y by transforming them back to source coordinates."""
    reverse_transformer = Transformer.from_crs(model_crs, source_crs, always_xy=True,)
    X_test, Y_test = reverse_transformer.transform(df_["X"].to_numpy(), df_["Y"].to_numpy(),)
    coordinates_match = (np.allclose(X_test, df_[X1].to_numpy(), atol=1e-8,)
        and np.allclose(Y_test,df_[Y1].to_numpy(),atol=1e-8,))
    if not coordinates_match:
        raise ValueError('Projected coordinates failed the round-trip check.')
    
    return True



# Determine CRS

In [ ]:
def project_coordinates(df, Longitude, Latitude, source_crs, model_crs,):
    if Longitude not in df.columns:
        raise ValueError(f"Missing longitude column: {Longitude}")

    if Latitude not in df.columns:
        raise ValueError(f"Missing latitude column: {Latitude}")  
    projected_df = df.copy()

    transformer = Transformer.from_crs(source_crs, model_crs, always_xy = True)
    projected_df['X'], projected_df['Y'] = transformer.transform(projected_df[Longitude].to_numpy(), projected_df[Latitude].to_numpy(),)
    
    # checking if projection transformation worked out
    validate_projection(projected_df, X1=Longitude, Y1=Latitude, source_crs=source_crs, model_crs=model_crs,)
    
    return projected_df

# Defining x,y,z coordinates and concentrations

In [ ]:
def create_well_points(df,X ,Y ,Z):
    X, Y, Z = df['X'],df['Y'],df['depth']

    well_points = np.column_stack(
        (
            X.to_numpy(),
            Y.to_numpy(),
            -Z.to_numpy())
    )
    return well_points

def add_contaminants(wells, data, list_of_contaminants):
    if len(data) != wells.n_points:
        raise ValueError(
            "The number of DataFrame rows does not match "
            "the number of PyVista points."
        )
    for contaminant in list_of_contaminants:
        if contaminant not in data.columns:
            raise ValueError(
                f"Contaminant '{contaminant}' is not in the original data."
            )
        wells[contaminant] = data[contaminant].to_numpy()
    return wells

### Diagnostics

In [ ]:
#Here but not very usefull
X, Y, Z = df['X'],df['Y'],df['depth']
def coordinate_diagnostics(X,Y,Z):   
    # calculating minimum, maximum, and span for each coordinate
    diagnostics = {
    #"X_min": X.min(),
    #"X_max": X.max(),
    "X_span": X.max() - X.min(),
    #"Y_min": Y.min(),
    #"Y_max": Y.max(),
    "Y_span":Y.max() - Y.min(),
    "Z_min": Z.min(),
    "Z_max": Z.max(),
    "Z_span": Z.max() - Z.min()}
    diagnostics= {k: float(v) for k, v in diagnostics.items()}
    return diagnostics
coordinate_diagnostics(X,Y,Z)

In [ ]:
# Very important for profiles!!! makes the distance matrix
def pairwise_horizontal_distances(df_, X, Y):
    coordinates = df_[[X, Y]].to_numpy(dtype=float)
    condensed_distance = pdist(coordinates, metric="euclidean",)
    distance_matrix = squareform(condensed_distance)

    return condensed_distance, distance_matrix

# Outputs

In [ ]:
# Defining x,y,z points using our df data and projected coordinates
well_points = create_well_points(df, X='X', Y='Y', Z='depth',)

# Transforming the x,y,z, array into a pvista array and assigning a COC as a scalar
pdata = pv.PolyData(well_points)
COC_list = ["PCB"] # COC means contaminant/s of concern
pdata = add_contaminants(
    wells=pdata,
    data=df,
    list_of_contaminants=COC_list,
)

# Adding samples at each well as speheres MAY NOT GO HERE LATER DOWN THE ROAD.. JUST IN ORDER OF ORIGINAL NOTEBOOK
sphere = pv.Sphere(radius=2, phi_resolution=10, theta_resolution=10)
pc = pdata.glyph(scale=False, geom=sphere, orient=False)
# pc.plot(cmap='seismic')

# Really just calling the distance matrix generator
condensed_distance, distance_matrix = (pairwise_horizontal_distances(df,X="X",Y="Y",))

# Using a shapefile to determine the model boundary
boundary_path = ("/mnt/c/Users/josem/OneDrive/Documents/Dummy_data/model_domain/shape_file/model_domain_2/model_domain_2.shp")
bounds = get_domain_bounds(boundaries=boundary_path,z_bounds=(-110, -20),model_crs=model_crs,)



